# Exercice 1 — Mini-RAG avec recherche BM25

Pipeline RAG complet avec recherche lexicale BM25. La recherche, le classement et la construction du prompt fonctionnent sans clé API.

**Modèle gratuit facultatif :** pour générer les réponses, Ollama peut exécuter localement `qwen2.5:3b` (aucun crédit OpenAI requis). Installer Ollama, puis lancer `ollama pull qwen2.5:3b` dans un terminal. Dans le notebook, installer ensuite `langchain-ollama`. La cellule de génération reste facultative ; le reste de l’exercice marche sans Ollama.

> Dépendance recherche : `pip install rank_bm25` (déjà listée dans `requirements.txt`).


In [2]:
import re
from rank_bm25 import BM25Okapi


## 1. Préparer le corpus

Chaque passage reçoit un identifiant qui servira de source à citer dans la réponse.


In [3]:
corpus = [
    "Le télétravail est autorisé deux jours par semaine, après validation du manager.",
    "Les congés payés doivent être posés au moins trois semaines à l’avance via l’intranet.",
    "Les notes de frais sont remboursées sous 30 jours si le justificatif est fourni.",
    "En cas d’arrêt maladie, l’employé doit transmettre l’arrêt sous 48 heures.",
    "Le matériel informatique est renouvelé tous les quatre ans.",
]

sources = [f"Passage {i}" for i in range(1, len(corpus) + 1)]
for source, passage in zip(sources, corpus):
    print(f"[{source}] {passage}")


[Passage 1] Le télétravail est autorisé deux jours par semaine, après validation du manager.
[Passage 2] Les congés payés doivent être posés au moins trois semaines à l’avance via l’intranet.
[Passage 3] Les notes de frais sont remboursées sous 30 jours si le justificatif est fourni.
[Passage 4] En cas d’arrêt maladie, l’employé doit transmettre l’arrêt sous 48 heures.
[Passage 5] Le matériel informatique est renouvelé tous les quatre ans.


## 2. Prétraiter et indexer avec BM25

Le prétraitement met le texte en minuscules, retire la ponctuation et enlève une liste simple de mots vides français. La même fonction sert à indexer les passages et à traiter les questions.


In [4]:
mots_vides = {
    "le", "la", "les", "l", "un", "une", "des", "du", "de", "d",
    "et", "est", "sont", "en", "au", "aux", "à", "a", "après",
    "pour", "par", "si", "sous", "via", "que", "qui", "doit",
    "être", "tous", "toutes", "cas",
}

def pretraiter(texte):
    """Normalise un texte et renvoie ses mots utiles pour BM25."""
    texte = texte.casefold()
    texte = re.sub(r"[^\w\s]", " ", texte, flags=re.UNICODE)
    tokens = texte.split()
    return [mot for mot in tokens if mot not in mots_vides]

corpus_tokenise = [pretraiter(passage) for passage in corpus]
index_bm25 = BM25Okapi(corpus_tokenise)

print(pretraiter("Les congés payés doivent être posés à l’avance !"))


['congés', 'payés', 'doivent', 'posés', 'avance']


## 3. Rechercher les passages pertinents

`rechercher(question, k=2)` renvoie les passages classés avec leur score BM25 et leur source. Un score plus élevé indique une meilleure correspondance lexicale. Un score nul signifie qu’aucun terme utile de la question ne correspond au passage.


In [5]:
def rechercher(question, k=2):
    termes_question = pretraiter(question)
    scores = index_bm25.get_scores(termes_question)
    classement = sorted(range(len(corpus)), key=lambda i: scores[i], reverse=True)

    return [
        {
            "passage": corpus[i],
            "source": sources[i],
            "score": float(scores[i]),
        }
        for i in classement[:k]
    ]

# Exemple de recherche
rechercher("Combien de jours de télétravail sont possibles ?", k=2)


[{'passage': 'Le télétravail est autorisé deux jours par semaine, après validation du manager.',
  'source': 'Passage 1',
  'score': 1.4350845252893227},
 {'passage': 'Les notes de frais sont remboursées sous 30 jours si le justificatif est fourni.',
  'source': 'Passage 3',
  'score': 0.33647223662121295}]

## 4. Construire le prompt du mini-RAG

Le prompt numérote les passages récupérés et donne leurs sources afin que le modèle puisse les citer. Il impose de s’en tenir au contexte et de reconnaître quand celui-ci ne suffit pas.


In [6]:
def construire_prompt(question, passages):
    contexte = "\n\n".join(
        f"[{i}] {element['passage']} (source : {element['source']})"
        for i, element in enumerate(passages, start=1)
    )
    if not contexte:
        contexte = "Aucun passage pertinent n’a été retrouvé."

    return f'''Tu es un assistant qui répond à une question à partir d’un contexte documentaire.
Réponds uniquement à partir du contexte ci-dessous. N’ajoute aucune information externe.
Cite la source utilisée dans ta réponse, par exemple [Passage 1].
Si le contexte ne permet pas de répondre, réponds exactement : « Je ne sais pas ».

Contexte :
{contexte}

Question : {question}
Réponse :'''


In [7]:
question = "Combien de jours de télétravail sont possibles ?"
passages = rechercher(question, k=2)
print(construire_prompt(question, passages))


Tu es un assistant qui répond à une question à partir d’un contexte documentaire.
Réponds uniquement à partir du contexte ci-dessous. N’ajoute aucune information externe.
Cite la source utilisée dans ta réponse, par exemple [Passage 1].
Si le contexte ne permet pas de répondre, réponds exactement : « Je ne sais pas ».

Contexte :
[1] Le télétravail est autorisé deux jours par semaine, après validation du manager. (source : Passage 1)

[2] Les notes de frais sont remboursées sous 30 jours si le justificatif est fourni. (source : Passage 3)

Question : Combien de jours de télétravail sont possibles ?
Réponse :


## 5. Tester les trois questions

Cette cellule affiche les passages BM25 et leur score pour chaque question. Les prompts peuvent ensuite être envoyés au modèle local facultatif présenté plus bas.


In [8]:
questions = [
    "Combien de jours de télétravail sont possibles ?",
    "Quel délai pour envoyer un arrêt maladie ?",
    "Quelle est la politique sur les véhicules de fonction ?",
]

for question in questions:
    print(f"\nQuestion : {question}")
    resultats = rechercher(question, k=2)
    for rang, resultat in enumerate(resultats, start=1):
        print(
            f"  {rang}. [{resultat['source']}] "
            f"score={resultat['score']:.3f} — {resultat['passage']}"
        )
    print("Prompt :")
    print(construire_prompt(question, resultats))



Question : Combien de jours de télétravail sont possibles ?
  1. [Passage 1] score=1.435 — Le télétravail est autorisé deux jours par semaine, après validation du manager.
  2. [Passage 3] score=0.336 — Les notes de frais sont remboursées sous 30 jours si le justificatif est fourni.
Prompt :
Tu es un assistant qui répond à une question à partir d’un contexte documentaire.
Réponds uniquement à partir du contexte ci-dessous. N’ajoute aucune information externe.
Cite la source utilisée dans ta réponse, par exemple [Passage 1].
Si le contexte ne permet pas de répondre, réponds exactement : « Je ne sais pas ».

Contexte :
[1] Le télétravail est autorisé deux jours par semaine, après validation du manager. (source : Passage 1)

[2] Les notes de frais sont remboursées sous 30 jours si le justificatif est fourni. (source : Passage 3)

Question : Combien de jours de télétravail sont possibles ?
Réponse :

Question : Quel délai pour envoyer un arrêt maladie ?
  1. [Passage 4] score=2.668 — En c

### Analyse des résultats

- **Télétravail** : le passage sur le télétravail contient les termes de la question et devrait arriver en tête. Il indique deux jours par semaine, après validation du manager.
- **Arrêt maladie** : la correspondance lexicale avec « arrêt maladie » permet de retrouver le passage indiquant un délai de 48 heures.
- **Véhicules de fonction** : ce sujet n’existe pas dans le corpus. BM25 peut tout de même renvoyer des passages avec un score nul ou faible, car il classe les documents disponibles même lorsqu’ils ne répondent pas à la question. Le prompt demande alors de répondre « Je ne sais pas » si le contexte ne suffit pas. En pratique, il est utile d’ajouter un seuil de pertinence avant de transmettre les passages au modèle.

### Question de réflexion : « Puis-je travailler depuis chez moi ? »

BM25 cherche les mots communs entre la question et les documents. La formulation ne contient ni « télétravail » ni « jours par semaine » : le passage pertinent risque donc d’obtenir un score nul ou faible. BM25 ne comprend pas automatiquement que « travailler depuis chez moi » signifie « télétravail ».

Sans embeddings, on peut améliorer la recherche avec une **expansion de requête** : remplacer ou compléter certains termes avant la recherche, par exemple `chez moi` → `télétravail`, `à domicile` → `télétravail`. On peut aussi utiliser un petit dictionnaire de synonymes, ou enrichir les passages avec leurs synonymes au moment de l’indexation. Ces méthodes restent lexicales et demandent de choisir les équivalences adaptées au domaine.

Le modèle local n’est pas nécessaire pour analyser les résultats : BM25 effectue la recherche, et le prompt encadre la réponse générée. Pour la question piège, un seuil de score peut éviter d’envoyer au modèle des passages sans rapport.


## 6. Générer une réponse avec un modèle gratuit (facultatif)

Ollama exécute le modèle sur votre ordinateur. Après avoir installé Ollama et téléchargé `qwen2.5:3b` (`ollama pull qwen2.5:3b`), installez le connecteur Python avec `%pip install langchain-ollama`, puis exécutez la cellule suivante. Aucun appel à OpenAI n’est effectué.


In [ ]:
# Cellule facultative : nécessite Ollama en cours d’exécution et le modèle qwen2.5:3b téléchargé.
from langchain_ollama import ChatOllama

modele = ChatOllama(model="qwen2.5:3b", temperature=0)

def repondre_avec_ollama(question, k=2):
    passages = rechercher(question, k=k)
    prompt = construire_prompt(question, passages)
    return modele.invoke(prompt).content

# Exemple :
question = "Combien de jours de télétravail sont possibles ?"
print(repondre_avec_ollama(question))
